# 04 — Parallel Infrastructure Benchmark
9 runs (N ∈ {1,2,4} × corpus ∈ {1k,5k,20k}). Generates Amdahl curve + latency plot.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
os.chdir('..')

In [ ]:
import json
import pandas as pd
from config import CORPUS_DIR, LOG_DIR, ACCURACY_DIR

# Load pre-built corpora from notebook 01
corpora = {}
for size in [1000, 5000, 20000]:
    path = os.path.join(CORPUS_DIR, f'corpus_{size}.json')
    with open(path) as f:
        corpora[size] = json.load(f)
    print(f'Loaded corpus_{size}: {len(corpora[size])} docs')

# Flatten to a single list ordered by size (benchmark slices by corpus_size)
all_docs = corpora[20000]  # largest; benchmark trims to corpus_size
print(f'Total docs available for benchmarking: {len(all_docs)}')

## Run Benchmark (9 runs)

In [ ]:
from parallel.benchmark import run_full_benchmark
from config import CORPUS_SIZES, PROCESS_COUNTS

# NOTE: N=1 runs first per corpus size — this is T_sequential for Amdahl
df_bench = run_full_benchmark(all_docs, corpus_sizes=CORPUS_SIZES, process_counts=PROCESS_COUNTS)
print(df_bench)

## Speedup Analysis

In [ ]:
from evaluation.amdahl import compute_speedup, fit_sequential_fraction

df_speedup = compute_speedup(df_bench)
S_fitted = fit_sequential_fraction(df_speedup)
print(f'Fitted sequential fraction S = {S_fitted:.3f}')
print(df_speedup[['corpus_size', 'n_processes', 'total_s', 'speedup_empirical']])

## Generate Plots

In [ ]:
from evaluation.reporter import plot_amdahl, plot_latency_vs_corpus, plot_accuracy_comparison, print_results_table

plot_amdahl(df_bench)
plot_latency_vs_corpus(df_bench)

In [ ]:
# Load accuracy results from notebooks 02 and 03
with open(os.path.join(ACCURACY_DIR, 'baseline_results.json')) as f:
    baseline_acc = json.load(f)
with open(os.path.join(ACCURACY_DIR, 'multihop_results.json')) as f:
    multihop_acc = json.load(f)

plot_accuracy_comparison(baseline_acc, multihop_acc)
print_results_table(baseline_acc, multihop_acc, df_bench)